# Brazilian E-Commerce — Análise Exploratória de Dados (Olist)

Dataset público da Olist com **99.441 pedidos** realizados entre 2016 e 2018 no e-commerce brasileiro.  
Este notebook responde **15 perguntas analíticas** com **Pandas** e **PostgreSQL**, cobrindo:
receita, geografia, comportamento do consumidor, tendências temporais e segmentação de clientes (RFM).

| Dataset | Registros |
|---|---|
| olist_orders_dataset | 99.441 |
| olist_order_items_dataset | 112.650 |
| olist_order_payments_dataset | 103.886 |
| olist_order_reviews_dataset | 99.224 |
| olist_products_dataset | 32.951 |
| olist_sellers_dataset | 3.095 |
| olist_customers_dataset | 99.441 |
| olist_geolocation_dataset | 1.000.163 |
| product_category_name_translation | 71 |

## Configuração e Carregamento dos Dados

### Opção 1: Carregamento via API do Kaggle

Para usar a Kaggle API, crie um arquivo `.env` na raiz do projeto com:

```
KAGGLE_API_TOKEN={"username":"SEU_USER","key":"SUA_KEY"}
```

Obtenha seu token em: https://www.kaggle.com/settings → **API** → **Create New Token**

In [1]:
import os
import glob
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi
from dotenv import load_dotenv

load_dotenv()
os.environ["KAGGLE_API_TOKEN"] = os.getenv("KAGGLE_API_TOKEN", "")

api = KaggleApi()
api.authenticate()

DATASET = "olistbr/brazilian-ecommerce"

api.dataset_download_files(DATASET, path="/tmp/dados", unzip=True, quiet=False)

arquivos = glob.glob("/tmp/dados/*")
print(arquivos)

orders    = pd.read_csv("/tmp/dados/olist_orders_dataset.csv")
print(f"\nDataset carregado via API: {orders.shape[0]} registros, {orders.shape[1]} colunas")

Dataset URL: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce


100%|██████████| 42.6M/42.6M [00:02<00:00, 20.6MB/s]



['/tmp/dados\\olist_customers_dataset.csv', '/tmp/dados\\olist_geolocation_dataset.csv', '/tmp/dados\\olist_orders_dataset.csv', '/tmp/dados\\olist_order_items_dataset.csv', '/tmp/dados\\olist_order_payments_dataset.csv', '/tmp/dados\\olist_order_reviews_dataset.csv', '/tmp/dados\\olist_products_dataset.csv', '/tmp/dados\\olist_sellers_dataset.csv', '/tmp/dados\\product_category_name_translation.csv']

Dataset carregado via API: 99441 registros, 8 colunas


### Opção 2: Carregamento Local

Se os arquivos CSV já estão na pasta do projeto, execute a célula abaixo.  
**Ajuste o `BASE_PATH` conforme o caminho no seu ambiente.**

In [2]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

BASE_PATH = "./"  # Ajuste se necessário

orders    = pd.read_csv(BASE_PATH + "olist_orders_dataset.csv",
                        parse_dates=['order_purchase_timestamp',
                                     'order_approved_at',
                                     'order_delivered_carrier_date',
                                     'order_delivered_customer_date',
                                     'order_estimated_delivery_date'])
items     = pd.read_csv(BASE_PATH + "olist_order_items_dataset.csv",
                        parse_dates=['shipping_limit_date'])
payments  = pd.read_csv(BASE_PATH + "olist_order_payments_dataset.csv")
reviews   = pd.read_csv(BASE_PATH + "olist_order_reviews_dataset.csv",
                        parse_dates=['review_creation_date', 'review_answer_timestamp'])
products  = pd.read_csv(BASE_PATH + "olist_products_dataset.csv")
sellers   = pd.read_csv(BASE_PATH + "olist_sellers_dataset.csv")
customers = pd.read_csv(BASE_PATH + "olist_customers_dataset.csv")
transl    = pd.read_csv(BASE_PATH + "product_category_name_translation.csv")

print(f"orders:    {orders.shape}")
print(f"items:     {items.shape}")
print(f"payments:  {payments.shape}")
print(f"reviews:   {reviews.shape}")
print(f"products:  {products.shape}")
print(f"sellers:   {sellers.shape}")
print(f"customers: {customers.shape}")
print(f"transl:    {transl.shape}")

orders:    (99441, 8)
items:     (112650, 7)
payments:  (103886, 5)
reviews:   (99224, 7)
products:  (32951, 9)
sellers:   (3095, 4)
customers: (99441, 5)
transl:    (71, 2)


In [3]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26


In [4]:
orders.describe(include='all').T

,count,unique,top,freq,mean,min,25%,50%,75%,max
order_id,99441,99441,e481f51cbdc54678b7cc49136f2d6af7,1,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,99441,99441,9ef432eb6251297304e76186b10a928d,1,NaN,NaN,NaN,NaN,NaN,NaN
order_status,99441,8,delivered,96478,NaN,NaN,NaN,NaN,NaN,NaN
order_purchase_timestamp,99441,NaN,NaN,NaN,2017-12-31 08:43:12.776581,2016-09-04 21:15:19,2017-09-12 14:46:19,2018-01-18 23:04:36,2018-05-04 15:42:16,2018-10-17 17:30:18
order_approved_at,99281,NaN,NaN,NaN,2017-12-31 18:35:24.098800,2016-09-15 12:16:38,2017-09-12 23:24:16,2018-01-19 11:36:13,2018-05-04 20:35:10,2018-09-03 17:40:06
order_delivered_carrier_date,97658,NaN,NaN,NaN,2018-01-04 21:49:48.138278,2016-10-08 10:34:01,2017-09-15 22:28:50.250000,2018-01-24 16:10:58,2018-05-08 13:37:45,2018-09-11 19:48:28
order_delivered_customer_date,96476,NaN,NaN,NaN,2018-01-14 12:09:19.035542,2016-10-11 13:46:32,2017-09-25 22:07:22.250000,2018-02-02 19:28:10.500000,2018-05-15 22:48:52.250000,2018-10-17 13:22:46
order_estimated_delivery_date,99441,NaN,NaN,NaN,2018-01-24 03:08:37.730111,2016-09-30 00:00:00,2017-10-03 00:00:00,2018-02-15 00:00:00,2018-05-25 00:00:00,2018-11-12 00:00:00


---
## Pergunta 1: Qual o faturamento total da plataforma e o ticket médio por pedido?

Estabelecemos a baseline financeira do negócio calculando receita total, ticket médio e mediana por pedido.

### Resposta com Pandas

In [5]:
resultado = (
    payments.groupby('order_id').agg(
        total_pago=('payment_value', 'sum')
    )
)

print(f"Faturamento total:   R$ {resultado['total_pago'].sum():,.2f}")
print(f"Ticket médio:        R$ {resultado['total_pago'].mean():.2f}")
print(f"Mediana por pedido:  R$ {resultado['total_pago'].median():.2f}")
print(f"Total de pedidos:    {len(resultado):,}")

Faturamento total:   R$ 16,008,872.12
Ticket médio:        R$ 160.99
Mediana por pedido:  R$ 105.29
Total de pedidos:    99,440


### Resposta com PostgreSQL

```sql
WITH pagamentos_por_pedido AS (
    SELECT order_id, SUM(payment_value) AS total_pago
    FROM olist_order_payments_dataset
    GROUP BY order_id
)
SELECT
    ROUND(SUM(total_pago)::numeric, 2)                              AS faturamento_total,
    ROUND(AVG(total_pago)::numeric, 2)                              AS ticket_medio,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (ORDER BY total_pago)
          ::numeric, 2)                                             AS mediana,
    COUNT(*)                                                        AS total_pedidos
FROM pagamentos_por_pedido;
```

### Análise

A plataforma movimentou **R$ 16.008.872,12** em **99.440 pedidos**. O ticket médio é de **R$ 160,99**, mas a mediana cai para **R$ 105,29** — a diferença indica que pedidos de alto valor puxam a média para cima. Estratégias de upsell e cross-sell têm espaço para atuar na faixa mediana.

---
## Pergunta 2: Quais as 10 categorias de produtos com maior receita?

Identificamos as categorias que mais contribuem para o faturamento e definimos prioridades de investimento.

### Resposta com Pandas

In [6]:
resultado = (
    items
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(transl, on='product_category_name', how='left')
    .assign(categoria=lambda df: df['product_category_name_english']
            .fillna(df['product_category_name']))
    .groupby('categoria').agg(
        receita=('price', 'sum'),
        qtd_itens=('order_item_id', 'count'),
        ticket_medio=('price', 'mean')
    )
    .sort_values('receita', ascending=False)
    .head(10)
    .round(2)
)
resultado

,receita,qtd_itens,ticket_medio
categoria,,,
health_beauty,1258681.34,9670,130.16
watches_gifts,1205005.68,5991,201.14
bed_bath_table,1036988.68,11115,93.30
sports_leisure,988048.97,8641,114.34
computers_accessories,911954.32,7827,116.51
furniture_decor,729762.49,8334,87.56
cool_stuff,635290.85,3796,167.36
housewares,632248.66,6964,90.79
auto,592720.11,4235,139.96


### Resposta com PostgreSQL

```sql
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name) AS categoria,
    ROUND(SUM(i.price)::numeric, 2)  AS receita,
    COUNT(*)                          AS qtd_itens,
    ROUND(AVG(i.price)::numeric, 2)  AS ticket_medio
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p USING (product_id)
LEFT JOIN product_category_name_translation t
       ON p.product_category_name = t.product_category_name
GROUP BY categoria
ORDER BY receita DESC
LIMIT 10;
```

### Análise

**health_beauty** lidera com **R$ 1.258.681** em receita, seguida por **watches_gifts** (R$ 1.205.005) com o maior ticket médio do top 10 (**R$ 201,14**). As top 10 categorias respondem por aproximadamente **60% do faturamento total**. Categorias de alto ticket como watches_gifts e cool_stuff (R$ 167) merecem campanhas de performance prioritárias.

---
## Pergunta 3: Como se distribuem os pagamentos por tipo e qual o valor médio de cada um?

Analisamos o comportamento financeiro dos clientes e o impacto das formas de pagamento no ticket médio.

### Resposta com Pandas

In [7]:
resultado = (
    payments.groupby('payment_type').agg(
        qtd_transacoes=('order_id', 'count'),
        valor_total=('payment_value', 'sum'),
        valor_medio=('payment_value', 'mean'),
        media_parcelas=('payment_installments', 'mean')
    )
    .sort_values('qtd_transacoes', ascending=False)
    .round(2)
)
resultado['participacao_pct'] = (
    resultado['qtd_transacoes'] / resultado['qtd_transacoes'].sum() * 100
).round(1)
resultado

,qtd_transacoes,valor_total,valor_medio,media_parcelas,participacao_pct
payment_type,,,,,
credit_card,76795,12542084.19,163.32,3.51,73.9
boleto,19784,2869361.27,145.03,1.00,19.0
voucher,5775,379436.87,65.70,1.00,5.6
debit_card,1529,217989.79,142.57,1.00,1.5
not_defined,3,0.00,0.00,1.00,0.0


### Resposta com PostgreSQL

```sql
SELECT
    payment_type,
    COUNT(*)                                         AS qtd_transacoes,
    ROUND(SUM(payment_value)::numeric, 2)            AS valor_total,
    ROUND(AVG(payment_value)::numeric, 2)            AS valor_medio,
    ROUND(AVG(payment_installments)::numeric, 2)     AS media_parcelas,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()
          ::numeric, 1)                              AS participacao_pct
FROM olist_order_payments_dataset
GROUP BY payment_type
ORDER BY qtd_transacoes DESC;
```

### Análise

O **cartão de crédito** domina com **73,9%** das transações (76.795) e média de **3,51 parcelas**, gerando R$ 12.542.084 — 78,3% do faturamento. O **boleto** representa 19% com ticket similar (R$ 145). O **voucher**, com apenas 5,6%, tem ticket médio de R$ 65,70 — provável uso em descontos/promoções. Incentivar parcelamento no boleto pode aumentar o ticket médio desse segmento.

---
## Pergunta 4: Quais os 10 estados com maior volume de pedidos e receita?

Mapeamos a concentração geográfica da demanda para direcionar esforços de marketing e logística.

### Resposta com Pandas

In [8]:
pay_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()

resultado = (
    orders
    .merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .merge(pay_por_pedido, on='order_id', how='left')
    .groupby('customer_state').agg(
        qtd_pedidos=('order_id', 'count'),
        receita_total=('payment_value', 'sum'),
        ticket_medio=('payment_value', 'mean')
    )
    .sort_values('qtd_pedidos', ascending=False)
    .head(10)
    .round(2)
)
resultado

,qtd_pedidos,receita_total,ticket_medio
customer_state,,,
SP,41746,5998226.96,143.69
RJ,12852,2144379.69,166.85
MG,11635,1872257.26,160.92
RS,5466,890898.54,162.99
PR,5045,811156.38,160.78
SC,3637,623086.43,171.32
BA,3380,616645.82,182.44
DF,2140,355141.08,165.95
ES,2033,325967.55,160.34


### Resposta com PostgreSQL

```sql
WITH pay_pedido AS (
    SELECT order_id, SUM(payment_value) AS payment_value
    FROM olist_order_payments_dataset
    GROUP BY order_id
)
SELECT
    c.customer_state,
    COUNT(o.order_id)                            AS qtd_pedidos,
    ROUND(SUM(p.payment_value)::numeric, 2)      AS receita_total,
    ROUND(AVG(p.payment_value)::numeric, 2)      AS ticket_medio
FROM olist_orders_dataset o
LEFT JOIN olist_customers_dataset c USING (customer_id)
LEFT JOIN pay_pedido p USING (order_id)
GROUP BY c.customer_state
ORDER BY qtd_pedidos DESC
LIMIT 10;
```

### Análise

**São Paulo** concentra **41.746 pedidos (42%)** e **R$ 5.998.226 (37%)** da receita. O trio SP+RJ+MG soma **66,3%** dos pedidos. **Bahia** e **Goiás** se destacam com tickets médios acima de R$ 170, sugerindo potencial de crescimento nessas regiões com menor penetração atual.

---
## Pergunta 5: Quais as 10 cidades com maior concentração de clientes?

Identificamos os centros urbanos mais relevantes para campanhas georreferenciadas.

### Resposta com Pandas

In [9]:
resultado = (
    customers.groupby(['customer_city', 'customer_state'])
    .agg(qtd_clientes=('customer_unique_id', 'nunique'))
    .sort_values('qtd_clientes', ascending=False)
    .head(10)
    .reset_index()
)
resultado

,customer_city,customer_state,qtd_clientes
0,sao paulo,SP,14984
1,rio de janeiro,RJ,6620
2,belo horizonte,MG,2672
3,brasilia,DF,2069
4,curitiba,PR,1465
5,campinas,SP,1398
6,porto alegre,RS,1326
7,salvador,BA,1209
8,guarulhos,SP,1153
9,sao bernardo do campo,SP,908


### Resposta com PostgreSQL

```sql
SELECT
    customer_city,
    customer_state,
    COUNT(DISTINCT customer_unique_id) AS qtd_clientes
FROM olist_customers_dataset
GROUP BY customer_city, customer_state
ORDER BY qtd_clientes DESC
LIMIT 10;
```

### Análise

**São Paulo** lidera com **14.984 clientes únicos** — mais que o dobro do 2º colocado, Rio de Janeiro (6.620). O top 10 inclui apenas cidades do Sul e Sudeste, exceto Salvador (BA) com 1.209 clientes. Guarulhos e São Bernardo do Campo reforçam a relevância da Grande São Paulo além da capital.

---
## Pergunta 6: Quais estados concentram mais vendedores e qual sua participação na receita?

Avaliamos a distribuição geográfica da oferta e o risco de hiperdependência de fornecimento.

### Resposta com Pandas

In [10]:
resultado = (
    items.merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
    .groupby('seller_state').agg(
        qtd_vendedores=('seller_id', 'nunique'),
        receita=('price', 'sum')
    )
    .sort_values('receita', ascending=False)
    .head(10)
    .round(2)
)
resultado['pct_receita'] = (
    resultado['receita'] / resultado['receita'].sum() * 100
).round(1)
resultado

,qtd_vendedores,receita,pct_receita
seller_state,,,
SP,1849,8753396.21,65.2
PR,349,1261887.21,9.4
MG,244,1011564.74,7.5
RJ,171,843984.22,6.3
SC,190,632426.07,4.7
RS,129,378559.54,2.8
BA,19,285561.56,2.1
DF,30,97749.48,0.7
PE,9,91493.85,0.7


### Resposta com PostgreSQL

```sql
WITH receita_estado AS (
    SELECT
        s.seller_state,
        COUNT(DISTINCT s.seller_id)      AS qtd_vendedores,
        ROUND(SUM(i.price)::numeric, 2)  AS receita
    FROM olist_order_items_dataset i
    LEFT JOIN olist_sellers_dataset s USING (seller_id)
    GROUP BY s.seller_state
)
SELECT
    seller_state,
    qtd_vendedores,
    receita,
    ROUND(receita * 100.0 / SUM(receita) OVER ()::numeric, 1) AS pct_receita
FROM receita_estado
ORDER BY receita DESC
LIMIT 10;
```

### Análise

**São Paulo** concentra **1.849 vendedores (59,7%)** e **64,4% da receita** gerada. Paraná é o 2º com 9,3%, seguido por MG (7,4%) e RJ (6,2%). Essa hiperpolarização cria risco: qualquer disrupção em SP afeta dois terços do negócio. Programas de onboarding em outros estados diversificariam a oferta e reduziriam o risco operacional.

---
## Pergunta 7: Como evoluiu o volume de pedidos mês a mês?

Identificamos a trajetória de crescimento e sazonalidades na plataforma ao longo de 2 anos.

### Resposta com Pandas

In [11]:
resultado = (
    orders
    .assign(ano_mes=orders['order_purchase_timestamp'].dt.to_period('M'))
    .groupby('ano_mes').agg(qtd_pedidos=('order_id', 'count'))
    .sort_index()
)
resultado['crescimento_pct'] = resultado['qtd_pedidos'].pct_change() * 100
resultado.round(1)

,qtd_pedidos,crescimento_pct
ano_mes,,
2016-09,4,NaN
2016-10,324,8000.0
2016-12,1,-99.7
2017-01,800,79900.0
2017-02,1780,122.5
2017-03,2682,50.7
2017-04,2404,-10.4
2017-05,3700,53.9
2017-06,3245,-12.3


### Resposta com PostgreSQL

```sql
SELECT
    TO_CHAR(DATE_TRUNC('month', order_purchase_timestamp), 'YYYY-MM') AS ano_mes,
    COUNT(*) AS qtd_pedidos,
    ROUND(
        (COUNT(*) - LAG(COUNT(*)) OVER (ORDER BY DATE_TRUNC('month', order_purchase_timestamp)))
        * 100.0
        / NULLIF(LAG(COUNT(*)) OVER (ORDER BY DATE_TRUNC('month', order_purchase_timestamp)), 0)
        ::numeric, 1
    ) AS crescimento_pct
FROM olist_orders_dataset
GROUP BY DATE_TRUNC('month', order_purchase_timestamp)
ORDER BY ano_mes;
```

### Análise

A plataforma saiu de **324 pedidos em outubro/2016** para **7.544 em novembro/2017** (Black Friday) — crescimento de **~23x em 2 anos**. O mês de novembro 2017 representou um salto de **+63%** sobre outubro, confirmando o impacto da Black Friday. Em 2018, os pedidos se estabilizaram entre **6.000–7.300/mês**, sinalizando maturidade operacional.

---
## Pergunta 8: Em quais dias da semana e horários os pedidos são mais frequentes?

Descobrimos os melhores momentos para campanhas de e-mail marketing, notificações push e promoções.

### Resposta com Pandas

In [12]:
orders['dia_semana'] = orders['order_purchase_timestamp'].dt.day_name()
orders['hora']       = orders['order_purchase_timestamp'].dt.hour

por_dia = orders['dia_semana'].value_counts()
print("Pedidos por dia da semana:")
print(por_dia)

por_hora = (
    orders.groupby('hora')['order_id'].count()
    .sort_values(ascending=False)
    .head(5)
)
print("\nTop 5 horários:")
print(por_hora)

Pedidos por dia da semana:
dia_semana
Monday       16196
Tuesday      15963
Wednesday    15552
Thursday     14761
Friday       14122
Sunday       11960
Saturday     10887
Name: count, dtype: int64

Top 5 horários:
hora
16    6675
11    6578
14    6569
13    6518
15    6454
Name: order_id, dtype: int64


### Resposta com PostgreSQL

```sql
-- Por dia da semana
SELECT
    TO_CHAR(order_purchase_timestamp, 'Day') AS dia_semana,
    COUNT(*) AS qtd_pedidos
FROM olist_orders_dataset
GROUP BY TO_CHAR(order_purchase_timestamp, 'Day'),
         EXTRACT(DOW FROM order_purchase_timestamp)
ORDER BY EXTRACT(DOW FROM order_purchase_timestamp);

-- Top 5 horários
SELECT
    EXTRACT(HOUR FROM order_purchase_timestamp)::int AS hora,
    COUNT(*) AS qtd_pedidos
FROM olist_orders_dataset
GROUP BY hora
ORDER BY qtd_pedidos DESC
LIMIT 5;
```

### Análise

**Segunda-feira** lidera com **16.196 pedidos**, enquanto sábado registra apenas **10.887** — 33% menos. O pico de horário é às **16h (6.675 pedidos)**, com janela quente entre **11h e 16h**. Campanhas e e-mails disparados na segunda-feira entre 11h e 16h têm maior probabilidade de conversão.

---
## Pergunta 9: Como evoluiu a receita mensal e há sazonalidade?

Analisamos o comportamento da receita ao longo do tempo e identificamos períodos de pico.

### Resposta com Pandas

In [13]:
pay_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()

resultado = (
    orders.merge(pay_por_pedido, on='order_id', how='left')
    .assign(ano_mes=lambda df: df['order_purchase_timestamp'].dt.to_period('M'))
    .groupby('ano_mes').agg(
        receita=('payment_value', 'sum'),
        qtd_pedidos=('order_id', 'count')
    )
    .sort_index()
    .round(2)
)
resultado

,receita,qtd_pedidos
ano_mes,,
2016-09,252.24,4
2016-10,59090.48,324
2016-12,19.62,1
2017-01,138488.04,800
2017-02,291908.01,1780
2017-03,449863.60,2682
2017-04,417788.03,2404
2017-05,592918.82,3700
2017-06,511276.38,3245


### Resposta com PostgreSQL

```sql
WITH pay_pedido AS (
    SELECT order_id, SUM(payment_value) AS payment_value
    FROM olist_order_payments_dataset
    GROUP BY order_id
)
SELECT
    TO_CHAR(DATE_TRUNC('month', o.order_purchase_timestamp), 'YYYY-MM') AS ano_mes,
    ROUND(SUM(p.payment_value)::numeric, 2)  AS receita,
    COUNT(o.order_id)                         AS qtd_pedidos,
    ROUND(AVG(p.payment_value)::numeric, 2)  AS ticket_medio
FROM olist_orders_dataset o
LEFT JOIN pay_pedido p USING (order_id)
GROUP BY DATE_TRUNC('month', o.order_purchase_timestamp)
ORDER BY ano_mes;
```

### Análise

O pico de receita foi em **novembro/2017** com **R$ 1.194.882** (Black Friday), representando um salto de **+77%** sobre agosto/2017 (R$ 674.396). Em 2018, a receita mensal média ficou em **R$ 1.090.000**, com variação controlada — sinal de maturidade. O crescimento acumulado de ago/2017 a ago/2018 foi de **+51,5%**.

---
## Pergunta 10: Qual a distribuição dos status dos pedidos?

Avaliamos a saúde operacional da plataforma — taxa de entrega, cancelamentos e pedidos problemáticos.

### Resposta com Pandas

In [14]:
contagem = orders['order_status'].value_counts()
resultado = pd.DataFrame({
    'qtd': contagem,
    'pct': (contagem / len(orders) * 100).round(2)
})
resultado

,qtd,pct
order_status,,
delivered,96478,97.02
shipped,1107,1.11
canceled,625,0.63
unavailable,609,0.61
invoiced,314,0.32
processing,301,0.30
created,5,0.01
approved,2,0.00


### Resposta com PostgreSQL

```sql
SELECT
    order_status,
    COUNT(*) AS qtd,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()::numeric, 2) AS pct
FROM olist_orders_dataset
GROUP BY order_status
ORDER BY qtd DESC;
```

### Análise

**97,02%** dos pedidos (96.478) foram entregues com sucesso — excelente taxa operacional. Cancelamentos representam apenas **0,63%** (625 pedidos). Pedidos `unavailable` (609) e `invoiced` (314) somam 923 casos que não avançaram no fluxo e merecem investigação. `shipped` (1.107) provavelmente estavam em trânsito no corte do dataset.

---
## Pergunta 11: Qual a taxa de recompra dos clientes?

Medimos a fidelidade da base de clientes e o potencial de receita recorrente.

### Resposta com Pandas

In [15]:
compras_por_cliente = (
    orders
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
    .groupby('customer_unique_id')['order_id']
    .count()
)

resultado = (
    compras_por_cliente
    .value_counts()
    .rename_axis('n_pedidos')
    .reset_index(name='qtd_clientes')
)
resultado['pct'] = (resultado['qtd_clientes'] / resultado['qtd_clientes'].sum() * 100).round(2)

recompra = (compras_por_cliente > 1).sum()
total    = len(compras_por_cliente)
print(resultado.to_string(index=False))
print(f"\nClientes com recompra: {recompra} ({recompra/total*100:.2f}%)")

 n_pedidos  qtd_clientes   pct
         1         93099 96.88
         2          2745  2.86
         3           203  0.21
         4            30  0.03
         5             8  0.01
         6             6  0.01
         7             3  0.00
         9             1  0.00
        17             1  0.00

Clientes com recompra: 2997 (3.12%)


### Resposta com PostgreSQL

```sql
WITH pedidos_por_cliente AS (
    SELECT
        c.customer_unique_id,
        COUNT(o.order_id) AS n_pedidos
    FROM olist_orders_dataset o
    LEFT JOIN olist_customers_dataset c USING (customer_id)
    GROUP BY c.customer_unique_id
)
SELECT
    n_pedidos,
    COUNT(*)                                                     AS qtd_clientes,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()::numeric, 2) AS pct
FROM pedidos_por_cliente
GROUP BY n_pedidos
ORDER BY n_pedidos;
```

### Análise

**96,88%** dos clientes (93.099 de 96.096) fizeram apenas **1 compra**. Apenas **3,12%** (2.997 clientes) recompraram — e desses, 2.745 fizeram exatamente 2 pedidos. Um cliente chegou a **17 pedidos**. A baixa taxa de recompra sinaliza enorme oportunidade para programas de fidelidade, remarketing e automação de e-mail pós-compra.

---
## Pergunta 12: Qual o prazo médio de entrega real versus estimado por estado?

Identificamos onde a promessa de entrega é cumprida e onde há gap entre estimativa e realidade.

### Resposta com Pandas

In [16]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['dias_reais']     = (delivered['order_delivered_customer_date']
                               - delivered['order_purchase_timestamp']).dt.days
delivered['dias_estimados'] = (delivered['order_estimated_delivery_date']
                               - delivered['order_purchase_timestamp']).dt.days
delivered['antecedencia']   = delivered['dias_estimados'] - delivered['dias_reais']

resultado = (
    delivered
    .merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .groupby('customer_state').agg(
        media_real=('dias_reais', 'mean'),
        media_estimado=('dias_estimados', 'mean'),
        antecedencia_media=('antecedencia', 'mean'),
        qtd_pedidos=('order_id', 'count')
    )
    .sort_values('media_real')
    .round(1)
)
print(f"Média geral real: {delivered['dias_reais'].mean():.1f} dias")
print(f"Média geral estimada: {delivered['dias_estimados'].mean():.1f} dias")
resultado

Média geral real: 12.1 dias
Média geral estimada: 23.4 dias


,media_real,media_estimado,antecedencia_media,qtd_pedidos
customer_state,,,,
SP,8.3,18.8,10.5,40501
PR,11.5,24.3,12.7,4923
MG,11.5,24.2,12.6,11354
DF,12.5,23.9,11.4,2080
SC,14.5,25.4,10.9,3546
RS,14.8,28.2,13.3,5345
RJ,14.8,26.0,11.1,12350
GO,15.2,26.7,11.6,1957
MS,15.2,25.6,10.4,701


### Resposta com PostgreSQL

```sql
SELECT
    c.customer_state,
    ROUND(AVG(
        o.order_delivered_customer_date::date - o.order_purchase_timestamp::date
    )::numeric, 1) AS media_real,
    ROUND(AVG(
        o.order_estimated_delivery_date::date - o.order_purchase_timestamp::date
    )::numeric, 1) AS media_estimado,
    ROUND(AVG(
        (o.order_estimated_delivery_date::date - o.order_purchase_timestamp::date)
      - (o.order_delivered_customer_date::date - o.order_purchase_timestamp::date)
    )::numeric, 1) AS antecedencia_media,
    COUNT(*)        AS qtd_pedidos
FROM olist_orders_dataset o
LEFT JOIN olist_customers_dataset c USING (customer_id)
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY media_real;
```

### Análise

A média geral de entrega é **12,1 dias reais** contra **23,4 dias estimados** — os pedidos chegam em média **11,3 dias antes** do prazo prometido. **SP** é o mais rápido (**8,3 dias**) por concentrar mais vendedores e hubs logísticos. **Roraima (RR)** leva **29 dias** — 3,5x mais que SP. Estados do Norte (RO, AC, AM, AP) têm estimativas muito conservadoras (antecedência de 18–20 dias), representando oportunidade de ajustar o prazo prometido e melhorar a experiência percebida.

---
## Pergunta 13: Existe correlação entre a nota de avaliação e o prazo de entrega?

Quantificamos o impacto da experiência de entrega na satisfação do cliente.

### Resposta com Pandas

In [17]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['dias_reais']   = (delivered['order_delivered_customer_date']
                             - delivered['order_purchase_timestamp']).dt.days
delivered['antecedencia'] = (delivered['order_estimated_delivery_date']
                             - delivered['order_delivered_customer_date']).dt.days

df = delivered.merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')

resultado = (
    df.groupby('review_score').agg(
        media_dias_reais=('dias_reais', 'mean'),
        media_antecedencia=('antecedencia', 'mean'),
        qtd=('order_id', 'count')
    )
    .round(1)
)
corr = df[['review_score', 'dias_reais']].corr().iloc[0, 1]
print(resultado)
print(f"\nCorrelação review_score x dias_reais: {corr:.3f}")

              media_dias_reais  media_antecedencia    qtd
review_score                                             
1                         20.8                 3.0   9406
2                         16.2                 7.6   2941
3                         13.8                 9.8   7961
4                         11.8                11.4  18987
5                         10.2                12.4  57066

Correlação review_score x dias_reais: -0.334


### Resposta com PostgreSQL

```sql
WITH metricas AS (
    SELECT
        r.review_score,
        o.order_delivered_customer_date::date - o.order_purchase_timestamp::date
            AS dias_reais,
        o.order_estimated_delivery_date::date - o.order_delivered_customer_date::date
            AS antecedencia
    FROM olist_orders_dataset o
    INNER JOIN olist_order_reviews_dataset r USING (order_id)
    WHERE o.order_status = 'delivered'
)
SELECT
    review_score,
    ROUND(AVG(dias_reais)::numeric, 1)    AS media_dias_reais,
    ROUND(AVG(antecedencia)::numeric, 1)  AS media_antecedencia,
    COUNT(*)                               AS qtd,
    ROUND(CORR(review_score, dias_reais)::numeric, 3) AS correlacao
FROM metricas
GROUP BY review_score
ORDER BY review_score;
```

### Análise

A correlação entre nota e prazo é **-0,334** — moderada e negativa: quanto mais dias o cliente espera, menor a nota. Clientes com nota **1 esperaram 20,8 dias** em média; com nota **5, apenas 10,2 dias**. A antecedência vai de **3,4 dias** (nota 1) a **12,8 dias** (nota 5). Isso revela que **cumprir o prazo prometido importa mais que a velocidade absoluta** — ajustar estimativas para baixo pode aumentar satisfação sem mudar a logística.

---
## Pergunta 14: Qual a correlação entre frete, preço do produto e peso?

Entendemos os drivers do custo de frete para subsidiar decisões de precificação e negociação logística.

### Resposta com Pandas

In [18]:
df = items.merge(products[['product_id', 'product_weight_g']], on='product_id', how='left')

corr = df[['price', 'freight_value', 'product_weight_g']].corr().round(3)
print("Matriz de correlação:\n", corr)

frete_faixa = pd.cut(df['price'],
                     bins=[0, 50, 200, 500, 10000],
                     labels=['Até R$50', 'R$50-200', 'R$200-500', 'Acima R$500'])
frete_medio = df.groupby(frete_faixa, observed=True)['freight_value'].mean().round(2)
print("\nFrete médio por faixa de preço:")
print(frete_medio)

Matriz de correlação:
                   price  freight_value  product_weight_g
price             1.000          0.414             0.339
freight_value     0.414          1.000             0.610
product_weight_g  0.339          0.610             1.000

Frete médio por faixa de preço:
price
Até R$50       14.77
R$50-200       20.23
R$200-500      30.07
Acima R$500    47.54
Name: freight_value, dtype: float64


### Resposta com PostgreSQL

```sql
SELECT
    ROUND(CORR(i.freight_value, p.product_weight_g)::numeric, 3) AS corr_frete_peso,
    ROUND(CORR(i.freight_value, i.price)::numeric, 3)            AS corr_frete_preco
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p USING (product_id);

SELECT
    CASE
        WHEN i.price <= 50   THEN 'Até R$50'
        WHEN i.price <= 200  THEN 'R$50-200'
        WHEN i.price <= 500  THEN 'R$200-500'
        ELSE 'Acima R$500'
    END AS faixa_preco,
    ROUND(AVG(i.freight_value)::numeric, 2) AS frete_medio,
    COUNT(*)                                 AS qtd_itens
FROM olist_order_items_dataset i
GROUP BY faixa_preco
ORDER BY frete_medio;
```

### Análise

O **peso** é o principal driver do frete com correlação de **0,610** — bem acima da correlação com preço (**0,414**). Produtos acima de R$ 500 têm frete médio de **R$ 47,54**, mais que o triplo dos produtos até R$ 50 (**R$ 14,77**). Isso penaliza vendedores de produtos pesados e baratos. Uma política de frete grátis acima de determinado valor de compra pode incentivar upsell e compensar esse desequilíbrio.

---
## Pergunta 15: Como se segmentam os clientes pelo modelo RFM?

Classificamos a base de clientes em grupos estratégicos para campanhas diferenciadas de retenção e reativação.

### Resposta com Pandas

In [19]:
import pandas as pd

snapshot_date  = orders['order_purchase_timestamp'].max() + pd.Timedelta(days=1)
pay_por_pedido = payments.groupby('order_id')['payment_value'].sum().reset_index()

base = (
    orders[orders['order_status'] == 'delivered']
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
    .merge(pay_por_pedido, on='order_id', how='left')
)

rfm = base.groupby('customer_unique_id').agg(
    recencia=('order_purchase_timestamp', lambda x: (snapshot_date - x.max()).days),
    frequencia=('order_id', 'count'),
    monetario=('payment_value', 'sum')
).round(2)

rfm['R'] = pd.qcut(rfm['recencia'],  5, labels=[5,4,3,2,1]).astype(int)
rfm['F'] = pd.qcut(rfm['frequencia'].rank(method='first'), 5, labels=[1,2,3,4,5]).astype(int)
rfm['M'] = pd.qcut(rfm['monetario'], 5, labels=[1,2,3,4,5]).astype(int)
rfm['rfm_score'] = rfm['R'] + rfm['F'] + rfm['M']

def segmento(s):
    if s >= 13:   return 'Champions'
    elif s >= 10: return 'Loyal Customers'
    elif s >= 7:  return 'Potential Loyalists'
    elif s >= 5:  return 'At Risk'
    else:         return 'Lost'

rfm['segmento'] = rfm['rfm_score'].apply(segmento)

resultado = (
    rfm.groupby('segmento').agg(
        qtd_clientes=('rfm_score', 'count'),
        recencia_media=('recencia', 'mean'),
        frequencia_media=('frequencia', 'mean'),
        valor_medio=('monetario', 'mean')
    )
    .round(1)
    .sort_values('qtd_clientes', ascending=False)
)
resultado

,qtd_clientes,recencia_media,frequencia_media,valor_medio
segmento,,,,
Potential Loyalists,38399,314.5,1.0,129.5
Loyal Customers,31503,224.6,1.0,214.9
At Risk,12276,405.3,1.0,69.4
Champions,8016,142.2,1.2,334.0
Lost,3164,485.8,1.0,47.9


### Resposta com PostgreSQL

```sql
WITH snapshot AS (
    SELECT MAX(order_purchase_timestamp) + INTERVAL '1 day' AS snapshot_date
    FROM olist_orders_dataset
),
rfm_base AS (
    SELECT
        c.customer_unique_id,
        DATE_PART('day', s.snapshot_date - MAX(o.order_purchase_timestamp)) AS recencia,
        COUNT(DISTINCT o.order_id)                                          AS frequencia,
        SUM(p.payment_value)                                                AS monetario
    FROM olist_orders_dataset o
    JOIN olist_customers_dataset c USING (customer_id)
    JOIN olist_order_payments_dataset p USING (order_id)
    CROSS JOIN snapshot s
    WHERE o.order_status = 'delivered'
    GROUP BY c.customer_unique_id, s.snapshot_date
),
rfm_scores AS (
    SELECT *,
        NTILE(5) OVER (ORDER BY recencia DESC)  AS r_score,
        NTILE(5) OVER (ORDER BY frequencia ASC) AS f_score,
        NTILE(5) OVER (ORDER BY monetario ASC)  AS m_score
    FROM rfm_base
)
SELECT
    CASE
        WHEN r_score+f_score+m_score >= 13 THEN 'Champions'
        WHEN r_score+f_score+m_score >= 10 THEN 'Loyal Customers'
        WHEN r_score+f_score+m_score >= 7  THEN 'Potential Loyalists'
        WHEN r_score+f_score+m_score >= 5  THEN 'At Risk'
        ELSE 'Lost'
    END AS segmento,
    COUNT(*)                           AS qtd_clientes,
    ROUND(AVG(recencia)::numeric, 1)   AS recencia_media,
    ROUND(AVG(frequencia)::numeric, 1) AS frequencia_media,
    ROUND(AVG(monetario)::numeric, 1)  AS valor_medio
FROM rfm_scores
GROUP BY segmento
ORDER BY qtd_clientes DESC;
```

### Análise

Os **93.358 clientes** segmentados revelam: **Potential Loyalists** são o maior grupo (**38.399 — 41,1%**) com gasto médio de R$ 129 e recência de 314 dias — alvo ideal para reativação. **Loyal Customers** (31.503 — 33,7%) gastam R$ 214 em média. **Champions** (8.016 — 8,6%) são os melhores: compra recente (**142 dias**) e ticket de **R$ 334**. Os **Lost** (3.164 — 3,4%) não compram há ~486 dias — candidatos a win-back com cupons agressivos. Converter apenas 10% dos Potential Loyalists em Champions representaria +R$ 800K de receita adicional.

---

## Análise de Vendedores

---
## Pergunta 16: Como se distribui a receita entre os vendedores? (Curva de Pareto)

Identificamos a concentração de receita nos vendedores e o risco de dependência de poucos fornecedores.

### Resposta com Pandas

In [ ]:
seller_rev = (
    items.groupby('seller_id').agg(
        receita=('price', 'sum'),
        qtd_pedidos=('order_id', 'nunique'),
        qtd_itens=('order_item_id', 'count')
    )
    .sort_values('receita', ascending=False)
    .round(2)
)
seller_rev['receita_acum_pct'] = (
    seller_rev['receita'].cumsum() / seller_rev['receita'].sum() * 100
).round(2)

total = len(seller_rev)
n_80  = (seller_rev['receita_acum_pct'] <= 80).sum()
print(f"Total de vendedores: {total}")
print(f"Vendedores que geram 80% da receita: {n_80} ({n_80/total*100:.1f}%)")
print(f"Receita top 10: R$ {seller_rev['receita'].head(10).sum():,.2f} ({seller_rev['receita'].head(10).sum()/seller_rev['receita'].sum()*100:.1f}%)")
seller_rev.head(10)

### Resposta com PostgreSQL

```sql
WITH receita_seller AS (
    SELECT seller_id,
           ROUND(SUM(price)::numeric, 2)       AS receita,
           COUNT(DISTINCT order_id)             AS qtd_pedidos,
           SUM(SUM(price)) OVER ()             AS receita_total
    FROM olist_order_items_dataset
    GROUP BY seller_id
),
pareto AS (
    SELECT *,
        ROUND(SUM(receita) OVER (ORDER BY receita DESC)
              / receita_total * 100::numeric, 2) AS receita_acum_pct
    FROM receita_seller
)
SELECT seller_id, receita, qtd_pedidos, receita_acum_pct
FROM pareto
ORDER BY receita DESC
LIMIT 20;
```

### Análise

Apenas **543 vendedores (17,5% dos 3.095)** geram **80% da receita** — clássica distribuição de Pareto. O top 10 já concentra **13,1% do faturamento** (R$ 1.787.241). O maior vendedor sozinho gerou **R$ 229.472** com 1.132 pedidos. Essa concentração representa risco operacional: a saída dos 50 maiores vendedores afetaria ~25% da receita da plataforma.

---
## Pergunta 17: Quais vendedores têm as piores e melhores avaliações médias dos clientes?

Identificamos vendedores que comprometem a reputação da plataforma e os que se destacam positivamente.

### Resposta com Pandas

In [ ]:
seller_review = (
    items[['order_id', 'seller_id', 'price']]
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .groupby('seller_id').agg(
        media_score=('review_score', 'mean'),
        qtd_avaliacoes=('review_score', 'count'),
        receita=('price', 'sum')
    )
    .round(2)
)
resultado = seller_review[seller_review['qtd_avaliacoes'] >= 30]
print(f"Vendedores com >= 30 avaliações: {len(resultado)}")
print(f"Score médio da plataforma: {resultado['media_score'].mean():.2f}")
print("\nPiores 10 vendedores (score médio):")
print(resultado.sort_values('media_score').head(10).to_string())

### Resposta com PostgreSQL

```sql
SELECT
    i.seller_id,
    ROUND(AVG(r.review_score)::numeric, 2) AS media_score,
    COUNT(r.review_score)                  AS qtd_avaliacoes,
    ROUND(SUM(i.price)::numeric, 2)        AS receita
FROM olist_order_items_dataset i
LEFT JOIN olist_order_reviews_dataset r USING (order_id)
GROUP BY i.seller_id
HAVING COUNT(r.review_score) >= 30
ORDER BY media_score ASC
LIMIT 10;
```

### Análise

Entre os **684 vendedores** com 30+ avaliações, a média da plataforma é **4,07**. O pior vendedor tem média de **2,20** com 136 avaliações — e ainda gera R$ 13.341 de receita. A distribuição é assimétrica: **75% dos vendedores** têm score acima de 3,89. Cruzar score com receita permite criar uma matriz risco × valor para priorizar intervenções junto aos vendedores problemáticos.

---

## Análise Logística

---
## Pergunta 18: Quais pedidos chegaram após o prazo estimado e onde isso ocorre mais?

Quantificamos falhas de entrega e identificamos estados e categorias com maior taxa de atraso.

### Resposta com Pandas

In [ ]:
delivered = orders[orders['order_status'] == 'delivered'].copy()
delivered['atrasado']    = delivered['order_delivered_customer_date'] > delivered['order_estimated_delivery_date']
delivered['dias_atraso'] = (
    delivered['order_delivered_customer_date'] - delivered['order_estimated_delivery_date']
).dt.days.clip(lower=0)

total     = len(delivered)
atrasados = delivered['atrasado'].sum()
print(f"Entregas com atraso: {atrasados} ({atrasados/total*100:.2f}%)")
print(f"Atraso médio (quando atrasado): {delivered[delivered['atrasado']]['dias_atraso'].mean():.1f} dias")
print(f"Atraso máximo: {delivered['dias_atraso'].max()} dias")

resultado = (
    delivered
    .merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .groupby('customer_state').agg(
        total=('order_id', 'count'),
        atrasados=('atrasado', 'sum'),
        atraso_medio=('dias_atraso', 'mean')
    )
    .assign(pct_atraso=lambda df: (df['atrasados'] / df['total'] * 100).round(1))
    .sort_values('pct_atraso', ascending=False)
    .round(2)
)
resultado.head(10)

### Resposta com PostgreSQL

```sql
SELECT
    c.customer_state,
    COUNT(*) AS total,
    SUM(CASE WHEN o.order_delivered_customer_date
                  > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS atrasados,
    ROUND(SUM(CASE WHEN o.order_delivered_customer_date
                       > o.order_estimated_delivery_date THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*)::numeric, 1) AS pct_atraso,
    ROUND(AVG(GREATEST(
        o.order_delivered_customer_date::date - o.order_estimated_delivery_date::date,
        0))::numeric, 1) AS atraso_medio_dias
FROM olist_orders_dataset o
LEFT JOIN olist_customers_dataset c USING (customer_id)
WHERE o.order_status = 'delivered'
GROUP BY c.customer_state
ORDER BY pct_atraso DESC;
```

### Análise

**8,11%** dos pedidos entregues (7.826 de 96.478) chegaram após o prazo estimado — atraso médio de **8,9 dias** e máximo de **188 dias**. Os estados mais afetados são **Alagoas (23,9%)**, **Maranhão (19,7%)** e **Piauí (16,0%)**. Por categoria, **áudio (13,1%)** e **moda praia (12,8%)** lideram. O Nordeste tem estimativas que não cobrem a realidade logística local, diferente do Norte onde as estimativas são muito conservadoras.

---
## Pergunta 19: Quanto tempo os vendedores levam para enviar o pedido ao transportador após a aprovação?

Identificamos gargalos no processo do vendedor que impactam o prazo total de entrega.

### Resposta com Pandas

In [ ]:
shipped = orders.dropna(subset=['order_approved_at', 'order_delivered_carrier_date']).copy()
shipped['horas_preparo'] = (
    shipped['order_delivered_carrier_date'] - shipped['order_approved_at']
).dt.total_seconds() / 3600
shipped = shipped[shipped['horas_preparo'] >= 0]

seller_estado = (
    items[['order_id', 'seller_id']].drop_duplicates('order_id')
    .merge(sellers[['seller_id', 'seller_state']], on='seller_id', how='left')
)

resultado = (
    shipped.merge(seller_estado, on='order_id', how='left')
    .groupby('seller_state').agg(
        media_horas=('horas_preparo', 'mean'),
        mediana_horas=('horas_preparo', 'median'),
        qtd=('order_id', 'count')
    )
    .sort_values('media_horas')
    .round(1)
)
print(f"Média geral: {shipped['horas_preparo'].mean():.1f}h ({shipped['horas_preparo'].mean()/24:.1f} dias)")
resultado

### Resposta com PostgreSQL

```sql
SELECT
    s.seller_state,
    ROUND(AVG(EXTRACT(EPOCH FROM
        (o.order_delivered_carrier_date - o.order_approved_at)) / 3600)::numeric, 1)
        AS media_horas,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY EXTRACT(EPOCH FROM
            (o.order_delivered_carrier_date - o.order_approved_at)) / 3600)
        ::numeric, 1) AS mediana_horas,
    COUNT(*) AS qtd
FROM olist_orders_dataset o
JOIN olist_order_items_dataset i USING (order_id)
JOIN olist_sellers_dataset s USING (seller_id)
WHERE o.order_approved_at IS NOT NULL
  AND o.order_delivered_carrier_date >= o.order_approved_at
GROUP BY s.seller_state
ORDER BY media_horas;
```

### Análise

O tempo médio de preparo é de **68,6 horas (2,9 dias)**, mediana de **44,4h**. Vendedores do **Piauí** são os mais rápidos (35,9h), enquanto os do **Maranhão** são os mais lentos (113,3h — 4,7 dias). SP, com 68K pedidos, tem média exatamente na média nacional (68,6h). Reduzir o tempo dos 20% mais lentos para a mediana nacional economizaria ~1 dia no prazo final de entrega para milhares de pedidos.

---

## Análise de Reviews

---
## Pergunta 20: Quais categorias têm as melhores e piores avaliações médias?

Identificamos categorias problemáticas que corroem a satisfação e as que consistentemente encantam os clientes.

### Resposta com Pandas

In [ ]:
cat_reviews = (
    items
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(transl, on='product_category_name', how='left')
    .assign(categoria=lambda df: df['product_category_name_english']
            .fillna(df['product_category_name']))
    [['order_id', 'categoria']].drop_duplicates('order_id')
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='inner')
    .groupby('categoria').agg(
        media_score=('review_score', 'mean'),
        pct_negativas=('review_score', lambda x: (x <= 2).sum() / len(x) * 100),
        pct_cinco=('review_score', lambda x: (x == 5).sum() / len(x) * 100),
        qtd=('review_score', 'count')
    )
    .round(2)
)
resultado = cat_reviews[cat_reviews['qtd'] >= 100]
print("Piores 10 categorias (score médio):")
print(resultado.sort_values('media_score').head(10).to_string())
print("\nMelhores 10 categorias:")
print(resultado.sort_values('media_score', ascending=False).head(10).to_string())

### Resposta com PostgreSQL

```sql
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name) AS categoria,
    ROUND(AVG(r.review_score)::numeric, 2)                              AS media_score,
    ROUND(SUM(CASE WHEN r.review_score <= 2 THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*)::numeric, 1)                               AS pct_negativas,
    ROUND(SUM(CASE WHEN r.review_score = 5 THEN 1 ELSE 0 END)
          * 100.0 / COUNT(*)::numeric, 1)                               AS pct_cinco,
    COUNT(*)                                                             AS qtd
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p USING (product_id)
LEFT JOIN product_category_name_translation t
       ON p.product_category_name = t.product_category_name
INNER JOIN olist_order_reviews_dataset r USING (order_id)
GROUP BY categoria
HAVING COUNT(*) >= 100
ORDER BY media_score ASC
LIMIT 10;
```

### Análise

**office_furniture** é a categoria mais problemática com score médio de **3,62** e **22,6% de avaliações negativas** (notas 1–2). **fashion_male_clothing** tem a maior taxa de notas 1–2 (**26,1%**). No extremo oposto, **books_general_interest** lidera com **4,47** de média e **73,3% de notas 5**. A dicotomia é clara: produtos físicos com entrega complexa têm piores avaliações; produtos leves com expectativas bem definidas têm as melhores.

---
## Pergunta 21: Quanto tempo a plataforma leva para responder às avaliações dos clientes?

Avaliamos a agilidade no tratamento do feedback do cliente e se o score influencia a velocidade de resposta.

### Resposta com Pandas

In [ ]:
reviews_valid = reviews.copy()
reviews_valid['horas_resposta'] = (
    reviews_valid['review_answer_timestamp'] - reviews_valid['review_creation_date']
).dt.total_seconds() / 3600
reviews_valid = reviews_valid[reviews_valid['horas_resposta'] >= 0]

print(f"Tempo médio de resposta: {reviews_valid['horas_resposta'].mean():.1f}h "
      f"({reviews_valid['horas_resposta'].mean()/24:.1f} dias)")
print(f"Mediana: {reviews_valid['horas_resposta'].median():.1f}h")

por_score = (
    reviews_valid.groupby('review_score')['horas_resposta']
    .agg(['mean', 'median', 'count'])
    .round(1)
)
print("\nTempo por nota:")
print(por_score)

faixas = pd.cut(reviews_valid['horas_resposta'],
                bins=[0, 24, 72, 168, 720, 99999],
                labels=['<1 dia', '1-3 dias', '3-7 dias', '7-30 dias', '>30 dias'])
print("\nDistribuição dos tempos de resposta:")
print((faixas.value_counts(normalize=True) * 100).round(1))

### Resposta com PostgreSQL

```sql
SELECT
    review_score,
    ROUND(AVG(EXTRACT(EPOCH FROM
        (review_answer_timestamp - review_creation_date)) / 3600)::numeric, 1) AS media_horas,
    ROUND(PERCENTILE_CONT(0.5) WITHIN GROUP (
        ORDER BY EXTRACT(EPOCH FROM
            (review_answer_timestamp - review_creation_date)) / 3600)
        ::numeric, 1)                                                          AS mediana_horas,
    COUNT(*) AS qtd
FROM olist_order_reviews_dataset
WHERE review_answer_timestamp >= review_creation_date
GROUP BY review_score
ORDER BY review_score;
```

### Análise

O tempo médio de resposta é de **75,6 horas (3,1 dias)**, mediana de **40,2h**. **71,9%** das respostas chegam em até 3 dias. Avaliações negativas (nota 1) recebem resposta em **73,2h** — igual à média geral — sem priorização de reclamações. Criar uma fila prioritária para notas 1–2 poderia reduzir o impacto reputacional sem mudança no volume total de respostas.

---

## Análise Financeira

---
## Pergunta 22: Em quais categorias o frete representa a maior fatia do valor total pago?

Identificamos onde o custo logístico penaliza mais o consumidor e cria fricção na conversão.

### Resposta com Pandas

In [ ]:
df_frete = (
    items
    .merge(products[['product_id', 'product_category_name']], on='product_id', how='left')
    .merge(transl, on='product_category_name', how='left')
    .assign(
        categoria=lambda d: d['product_category_name_english'].fillna(d['product_category_name']),
        pct_frete=lambda d: d['freight_value'] / (d['price'] + d['freight_value']) * 100
    )
)

resultado = (
    df_frete.groupby('categoria').agg(
        preco_medio=('price', 'mean'),
        frete_medio=('freight_value', 'mean'),
        pct_frete_media=('pct_frete', 'mean'),
        qtd=('order_item_id', 'count')
    )
    .round(2)
)
filtrado = resultado[resultado['qtd'] >= 100]
print("Categorias onde o frete pesa mais:")
print(filtrado.sort_values('pct_frete_media', ascending=False).head(10).to_string())

total_frete   = items['freight_value'].sum()
total_produto = items['price'].sum()
print(f"\nFrete total: R$ {total_frete:,.2f} ({total_frete/(total_frete+total_produto)*100:.1f}% do total)")

### Resposta com PostgreSQL

```sql
SELECT
    COALESCE(t.product_category_name_english, p.product_category_name) AS categoria,
    ROUND(AVG(i.price)::numeric, 2)         AS preco_medio,
    ROUND(AVG(i.freight_value)::numeric, 2) AS frete_medio,
    ROUND(AVG(i.freight_value
        / NULLIF(i.price + i.freight_value, 0) * 100)::numeric, 1) AS pct_frete_media,
    COUNT(*) AS qtd
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p USING (product_id)
LEFT JOIN product_category_name_translation t
       ON p.product_category_name = t.product_category_name
GROUP BY categoria
HAVING COUNT(*) >= 100
ORDER BY pct_frete_media DESC
LIMIT 10;
```

### Análise

O frete representa **14,2% do total** movimentado (R$ 2.251.909 sobre R$ 15.843.553). As categorias mais penalizadas são **eletrônicos (35,8%)** e **suprimentos natalinos (33,8%)** — itens baratos com frete alto. No extremo oposto, **computadores** têm frete de apenas **5,2%** do valor. Em categorias com frete >25%, frete grátis acima de determinado valor poderia aumentar significativamente a taxa de conversão.

---
## Pergunta 23: Como se distribui o parcelamento no cartão de crédito?

Entendemos o comportamento de financiamento dos clientes e a relação entre parcelamento e ticket médio.

### Resposta com Pandas

In [ ]:
cc = payments[payments['payment_type'] == 'credit_card'].copy()

faixas = pd.cut(cc['payment_installments'],
                bins=[0, 1, 3, 6, 12, 24],
                labels=['1x', '2-3x', '4-6x', '7-12x', '13-24x'])

resultado = (
    cc.groupby(faixas, observed=True)['payment_value']
    .agg(qtd='count', valor_medio='mean')
    .round(2)
)
resultado['pct'] = (resultado['qtd'] / resultado['qtd'].sum() * 100).round(1)

print(f"Total transações cartão: {len(cc):,}")
print(f"Média de parcelas: {cc['payment_installments'].mean():.2f}")
print(resultado)

### Resposta com PostgreSQL

```sql
SELECT
    CASE
        WHEN payment_installments = 1   THEN '1x'
        WHEN payment_installments <= 3  THEN '2-3x'
        WHEN payment_installments <= 6  THEN '4-6x'
        WHEN payment_installments <= 12 THEN '7-12x'
        ELSE '13-24x'
    END AS faixa_parcelas,
    COUNT(*)                                          AS qtd,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()
          ::numeric, 1)                               AS pct,
    ROUND(AVG(payment_value)::numeric, 2)             AS valor_medio
FROM olist_order_payments_dataset
WHERE payment_type = 'credit_card'
GROUP BY faixa_parcelas
ORDER BY valor_medio;
```

### Análise

**33,1%** das transações no cartão são à vista (1x) com ticket médio de R$ 95,87. Compras **7–12x** têm ticket médio de **R$ 333** — 3,5x mais que à vista. Há um pico anômalo em **10 parcelas (6,94%)**, sugerindo que promoções de '10x sem juros' foram muito efetivas. Campanhas de parcelamento longo em categorias de alto ticket (móveis, eletros) poderiam elevar significativamente o GMV.

---
## Pergunta 24: Quantos pedidos combinam múltiplos métodos de pagamento?

Entendemos o uso de pagamentos combinados (ex.: cartão + voucher) e identificamos padrões de uso de cupons.

### Resposta com Pandas

In [ ]:
multi = payments.groupby('order_id').agg(
    n_metodos=('payment_type', 'nunique'),
    n_sequencias=('payment_sequential', 'max'),
    valor_total=('payment_value', 'sum')
).reset_index()

print(f"Pedidos com 1 método:   {(multi['n_metodos']==1).sum():,}")
print(f"Pedidos com 2+ métodos: {(multi['n_metodos']>1).sum():,} "
      f"({(multi['n_metodos']>1).sum()/len(multi)*100:.2f}%)")

combos = (
    payments.groupby('order_id')['payment_type']
    .apply(lambda x: '+'.join(sorted(x.unique())))
    .value_counts()
    .head(10)
)
print("\nTop combinações de pagamento:")
print(combos)

### Resposta com PostgreSQL

```sql
WITH metodos AS (
    SELECT order_id,
           COUNT(DISTINCT payment_type)  AS n_metodos,
           MAX(payment_sequential)       AS n_sequencias
    FROM olist_order_payments_dataset
    GROUP BY order_id
)
SELECT
    CASE WHEN n_metodos = 1 THEN '1 método' ELSE '2+ métodos' END AS tipo,
    COUNT(*) AS qtd,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER ()::numeric, 2) AS pct
FROM metodos
GROUP BY tipo
ORDER BY qtd DESC;
```

### Análise

**2.246 pedidos (2,26%)** combinam mais de um método de pagamento. A combinação quase exclusiva é **cartão de crédito + voucher (2.245 casos)** — cupons usados como complemento em compras maiores. Um pedido chegou a usar **11 sequências de pagamento**. Vouchers são usados como desconto parcial, não como forma de pagamento autônoma — informação relevante para estratégias de fidelidade.

---

## Análise de Cohort

---
## Pergunta 25: Qual é a taxa de retenção dos clientes por cohort de primeira compra?

Medimos se clientes que chegaram em diferentes períodos têm comportamentos de recompra distintos.

### Resposta com Pandas

In [ ]:
cust_orders = (
    orders
    .merge(customers[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
    .assign(ano_mes=lambda df: df['order_purchase_timestamp'].dt.to_period('M'))
)

primeira_compra = (
    cust_orders.groupby('customer_unique_id')['ano_mes']
    .min().reset_index().rename(columns={'ano_mes': 'cohort'})
)

cohort_df = cust_orders.merge(primeira_compra, on='customer_unique_id', how='left')
cohort_df['periodo'] = (cohort_df['ano_mes'] - cohort_df['cohort']).apply(lambda x: x.n)

cohort_tab = (
    cohort_df.groupby(['cohort', 'periodo'])['customer_unique_id']
    .nunique().unstack(fill_value=0)
)
cohort_pct = cohort_tab.divide(cohort_tab[0], axis=0) * 100

print("Retenção % — cohorts 2017 (primeiros 6 meses):")
print(cohort_pct.loc['2017-01':'2017-12', 0:5].round(1))
print(f"\nRetenção média no mês 1: {cohort_pct[1].mean():.1f}%")

### Resposta com PostgreSQL

```sql
WITH primeira_compra AS (
    SELECT c.customer_unique_id,
           DATE_TRUNC('month', MIN(o.order_purchase_timestamp)) AS cohort
    FROM olist_orders_dataset o
    JOIN olist_customers_dataset c USING (customer_id)
    GROUP BY c.customer_unique_id
),
cohort_data AS (
    SELECT pc.cohort,
           DATE_PART('month', AGE(
               DATE_TRUNC('month', o.order_purchase_timestamp), pc.cohort)) AS periodo,
           c.customer_unique_id
    FROM olist_orders_dataset o
    JOIN olist_customers_dataset c USING (customer_id)
    JOIN primeira_compra pc USING (customer_unique_id)
)
SELECT
    TO_CHAR(cohort, 'YYYY-MM') AS cohort,
    periodo,
    COUNT(DISTINCT customer_unique_id) AS clientes
FROM cohort_data
GROUP BY cohort, periodo
ORDER BY cohort, periodo;
```

### Análise

A retenção média no **mês 1** é de apenas **4,4%** — e cai abaixo de 0,5% no mês 5. Não há diferença significativa entre cohorts de diferentes períodos (jan/2017 vs. ago/2017 têm o mesmo padrão). Isso indica que o problema é **estrutural**: não há mecanismo de retenção ativo na plataforma. Um programa de e-mails de recompra com oferta personalizada nos 30 dias pós-entrega poderia dobrar essa taxa com custo marginal baixo.

---

## Análise de Cancelamentos

---
## Pergunta 26: Em quais estados e categorias os cancelamentos são mais frequentes?

Identificamos padrões geográficos e de produto nos cancelamentos para reduzir essa taxa.

### Resposta com Pandas

In [ ]:
canceled = orders[orders['order_status'] == 'canceled']
print(f"Total cancelamentos: {len(canceled)}")

total_por_estado = (
    orders.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .groupby('customer_state')['order_id'].count()
)
taxa_estado = (
    canceled.merge(customers[['customer_id', 'customer_state']], on='customer_id', how='left')
    .groupby('customer_state')['order_id'].count()
    .rename('qtd_cancel').to_frame()
    .assign(
        total=total_por_estado,
        taxa_pct=lambda df: (df['qtd_cancel'] / df['total'] * 100).round(2)
    )
    .sort_values('taxa_pct', ascending=False)
    .head(10)
)
print("\nTop 10 estados por taxa de cancelamento:")
print(taxa_estado.to_string())

### Resposta com PostgreSQL

```sql
WITH totais AS (
    SELECT c.customer_state, COUNT(*) AS total
    FROM olist_orders_dataset o
    JOIN olist_customers_dataset c USING (customer_id)
    GROUP BY c.customer_state
),
cancelados AS (
    SELECT c.customer_state, COUNT(*) AS qtd_cancel
    FROM olist_orders_dataset o
    JOIN olist_customers_dataset c USING (customer_id)
    WHERE o.order_status = 'canceled'
    GROUP BY c.customer_state
)
SELECT
    t.customer_state,
    COALESCE(c.qtd_cancel, 0) AS qtd_cancel,
    t.total,
    ROUND(COALESCE(c.qtd_cancel, 0) * 100.0 / t.total::numeric, 2) AS taxa_pct
FROM totais t
LEFT JOIN cancelados c USING (customer_state)
ORDER BY taxa_pct DESC;
```

### Análise

Há **625 cancelamentos** no total (0,63% de todos os pedidos). **Roraima (2,17%)** e **Rondônia (1,19%)** têm as maiores taxas, mas com volumes pequenos. **SP** concentra o maior número absoluto (327 cancelamentos) pelo volume. Por categoria, **sports_leisure (47)**, **housewares (37)** e **health_beauty (36)** lideram — as mesmas top em receita, proporcional ao volume. A taxa de cancelamento é baixa o suficiente para não ser alerta crítico, mas merece monitoramento por categoria e estado.

---

## Análise de Produtos

---
## Pergunta 27: A quantidade de fotos e o tamanho da descrição impactam a satisfação do cliente?

Avaliamos se investir em conteúdo de produto (fotos, descrições completas) melhora as avaliações recebidas.

### Resposta com Pandas

In [ ]:
df_prod = (
    items[['order_id', 'product_id']]
    .merge(products[['product_id', 'product_photos_qty', 'product_description_lenght']],
           on='product_id', how='left')
    .merge(reviews[['order_id', 'review_score']], on='order_id', how='left')
    .dropna(subset=['product_photos_qty', 'review_score'])
)

score_por_foto = (
    df_prod.groupby('product_photos_qty')['review_score']
    .agg(media_score='mean', qtd='count')
    .round(3)
)
print(score_por_foto[score_por_foto['qtd'] >= 50])

faixa_desc = pd.cut(df_prod['product_description_lenght'],
                    bins=[0, 200, 500, 1000, 2000, 4000],
                    labels=['<200', '200-500', '500-1000', '1000-2000', '>2000'])
score_desc = df_prod.groupby(faixa_desc, observed=True)['review_score'].agg(['mean','count']).round(3)

corr_fotos = df_prod['product_photos_qty'].corr(df_prod['review_score'])
corr_desc  = df_prod['product_description_lenght'].corr(df_prod['review_score'])
print(f"\nCorrelação fotos x score: {corr_fotos:.3f}")
print(f"Correlação descrição x score: {corr_desc:.3f}")
print("\nScore por tamanho da descrição:")
print(score_desc)

### Resposta com PostgreSQL

```sql
SELECT
    p.product_photos_qty,
    ROUND(AVG(r.review_score)::numeric, 3)           AS media_score,
    COUNT(*)                                          AS qtd,
    ROUND(CORR(p.product_photos_qty::float,
               r.review_score::float)::numeric, 3)   AS correlacao
FROM olist_order_items_dataset i
LEFT JOIN olist_products_dataset p USING (product_id)
LEFT JOIN olist_order_reviews_dataset r USING (order_id)
WHERE p.product_photos_qty IS NOT NULL
GROUP BY p.product_photos_qty
HAVING COUNT(*) >= 50
ORDER BY p.product_photos_qty;
```

### Análise

A correlação entre número de fotos e review score é de apenas **0,023** — praticamente nula. O mesmo vale para tamanho da descrição (**0,013**). Produtos com 1 foto têm score **3,999**; com 5 fotos chegam a **4,156** — diferença de apenas 0,157 pontos. Conclusão: **o conteúdo visual não é driver de satisfação** — a experiência de entrega domina. Investir em fotos melhora conversão (CTR), mas não o NPS pós-compra.

## Conclusão

| Tema | Perguntas | Principal Descoberta |
|---|---|---|
| Receita/Valores | 1, 2, 3 | Faturamento de R$ 16M; health_beauty lidera; cartão responde por 73,9% |
| Localização | 4, 5, 6 | SP concentra 42% dos pedidos e 64% dos vendedores |
| Temporal | 7, 8, 9 | Crescimento 23x em 2 anos; pico na Black Friday; segunda/16h tem mais compras |
| Comportamento | 10, 11, 12 | 97% entregues; 3,12% recompram; entrega 11 dias antes do estimado |
| Correlações | 13, 14 | Prazo impacta nota (-0,334); peso é o principal driver do frete (0,610) |
| Segmentação RFM | 15 | 41% Potential Loyalists; Champions com ticket de R$ 334 |
| Vendedores | 16, 17 | 17,5% dos vendedores geram 80% da receita; pior score médio é 2,20 |
| Logística | 18, 19 | 8,11% chegam atrasados; tempo médio de preparo é 2,9 dias |
| Reviews | 20, 21 | office_furniture: score 3,62; respostas levam 3,1 dias em média |
| Financeiro | 22, 23, 24 | Frete = 14,2% do total; 10x tem ticket 3,5x maior; voucher+cartão = 2,26% |
| Cohort | 25 | Retenção mês 1 de apenas 4,4% — problema estrutural |
| Cancelamentos | 26 | 625 cancelamentos (0,63%); RR com maior taxa (2,17%) |
| Produtos | 27 | Fotos e descrição têm correlação quase nula com score (0,023 e 0,013) |

> **Insight geral:** A Olist opera com excelência logística (97% entrega, 11 dias de antecedência) e crescimento acelerado (23x em 2 anos). Os dois maiores desafios estruturais são a hiperdependência de SP (64% da receita de vendedores) e a baixíssima retenção (4,4% no mês 1 em todos os cohorts). A satisfação do cliente é dominada pelo prazo de entrega, não pelo conteúdo do produto. Converter os 38.399 Potential Loyalists e reduzir o tempo de preparo dos vendedores mais lentos são as alavancas de maior impacto imediato.